# EP14 — Search Algorithms: Completeness & Optimality
**COMPSCI 713 · S1 2025 Q5 · 3 marks**

| Part | Topic | Marks |
|------|-------|-------|
| a | Define completeness and optimality | 1 |
| b | DFS and Iterative Deepening — complete? optimal? why? | 2 |

---

## Part (a) — Definitions [1 mark]

### Completeness
A search algorithm is **complete** if it is *guaranteed to find a solution whenever one exists*.

- If a goal node exists in the search space, a complete algorithm will always find it (given sufficient resources).
- If no solution exists, a complete algorithm must correctly report failure (not loop forever).

### Optimality
A search algorithm is **optimal** if it is *guaranteed to find the best (minimum-cost) solution* among all possible solutions.

- In unweighted graphs/trees: optimal = finds the shallowest goal node (fewest steps).
- In weighted graphs: optimal = finds the path with the lowest total cost.

---

## Part (b) — DFS and Iterative Deepening [2 marks]

### Depth-First Search (DFS)

**Complete?** — **No (in infinite/cyclic spaces). Conditionally yes in finite acyclic trees.**

Why: DFS commits to exploring one branch completely before backtracking. In an infinite tree (unbounded depth), DFS can follow an infinite branch and never return — it will never find a goal node on a different branch. It can also loop forever in graphs with cycles unless cycle detection is used.

*Condition for completeness: the search tree is finite and acyclic.*

**Optimal?** — **No.**

Why: DFS returns the *first* solution it finds, which may be deep in the tree. A shallower (cheaper) solution on another branch is never explored. DFS has no preference for shorter paths.

---

### Iterative Deepening (IDS / IDDFS)

**Complete?** — **Yes.**

Why: IDS systematically explores all nodes at depth 0, then depth 1, then depth 2, etc. Since it exhaustively covers every depth level before going deeper, it cannot miss a goal node that exists at any finite depth. Even in trees with infinite depth, it will find a goal at depth d after completing depth d−1.

**Optimal?** — **Yes (in trees with uniform step costs).**

Why: Because IDS explores all nodes at depth d before any node at depth d+1, the first solution it finds is guaranteed to be at the shallowest depth. In uniform-cost settings, shallowest = cheapest. (Note: not optimal for non-uniform step costs without modification.)

---

### Summary Table

| Algorithm | Complete? | Optimal? | Why complete? | Why optimal? |
|-----------|-----------|----------|---------------|--------------|
| DFS | No (infinite trees) | No | Can follow infinite branch | Returns first found, not shallowest |
| IDS | Yes | Yes (uniform cost) | Covers all depths level by level | Finds shallowest solution first |

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque

# =============================================================
# Build a simple tree and compare DFS vs IDS
# Shows visit order and whether optimal solution is found
# =============================================================

# Tree represented as adjacency: parent → children
# Nodes: 0 is root; goal nodes are labelled 'G'
#
#         0
#        / \
#       1   2
#      / \ / \
#     3  4 5  6
#    /  <--- goal at depth 3 only via node 3
#   7*  8   9  10*
# (* = goal)

tree = {
    0: [1, 2],
    1: [3, 4],
    2: [5, 6],
    3: [7, 8],
    4: [],
    5: [9],
    6: [10],
    7: [], 8: [], 9: [], 10: []
}
goals = {10}   # goal is on the RIGHT side, depth 3
# Note: DFS (left-first) will explore 0→1→3→7 first (dead end),
# then 8, then backtrack to 4, then 2→5→9, then 6→10 (goal)
# Shallowest goal is at depth 3 — both methods should find it
# but DFS finds it last while IDS finds it first at depth 3

def dfs(tree, start, goals):
    """DFS — returns (path_to_goal, visit_order)"""
    stack = [(start, [start])]
    visited = []
    while stack:
        node, path = stack.pop()
        visited.append(node)
        if node in goals:
            return path, visited
        for child in reversed(tree.get(node, [])):  # reversed for left-first
            stack.append((child, path + [child]))
    return None, visited

def ids(tree, start, goals, max_depth=10):
    """Iterative Deepening — returns (path_to_goal, visit_order_per_iteration)"""
    all_visits = []
    for depth_limit in range(max_depth + 1):
        visits_this = []
        result = _dls(tree, start, goals, depth_limit, [start], visits_this)
        all_visits.append((depth_limit, visits_this))
        if result:
            return result, all_visits
    return None, all_visits

def _dls(tree, node, goals, limit, path, visits):
    visits.append(node)
    if node in goals:
        return path
    if limit == 0:
        return None
    for child in tree.get(node, []):
        result = _dls(tree, child, goals, limit-1, path+[child], visits)
        if result:
            return result
    return None

dfs_path, dfs_visits = dfs(tree, 0, goals)
ids_path, ids_iterations = ids(tree, 0, goals)

print('=== Search Comparison ===')
print(f'Goal nodes: {goals}')
print()
print(f'DFS visit order: {dfs_visits}')
print(f'DFS solution path: {dfs_path}  (depth {len(dfs_path)-1})')
print()
print('IDS iterations:')
for depth, visits in ids_iterations:
    print(f'  Depth {depth}: visited {visits}')
if ids_path:
    print(f'IDS solution path: {ids_path}  (depth {len(ids_path)-1})')

In [ ]:
# =============================================================
# Visualisation: tree with DFS and IDS visit order highlighted
# =============================================================

# Node positions for drawing
pos = {
    0:  (4.0, 6.0),
    1:  (2.0, 4.5),  2:  (6.0, 4.5),
    3:  (1.0, 3.0),  4:  (3.0, 3.0),
    5:  (5.0, 3.0),  6:  (7.0, 3.0),
    7:  (0.5, 1.5),  8:  (1.5, 1.5),
    9:  (4.5, 1.5),  10: (7.5, 1.5),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, visit_list, path, title, colour in [
    (axes[0], dfs_visits, dfs_path, 'DFS — visit order', '#e74c3c'),
    (axes[1],
     [v for _, vlist in ids_iterations for v in vlist],
     ids_path,
     'IDS — visit order (all iterations)', '#2980b9'),
]:
    # Draw edges
    for parent, children in tree.items():
        px, py = pos[parent]
        for child in children:
            cx, cy = pos[child]
            ax.plot([px, cx], [py, cy], 'k-', alpha=0.3, zorder=1)

    # Draw nodes
    for node, (nx, ny) in pos.items():
        visit_idx = next((i for i, v in enumerate(visit_list) if v == node), None)
        is_goal   = node in goals
        is_path   = path and node in path

        if is_goal:
            node_colour = '#2ecc71'
        elif is_path:
            node_colour = colour
        elif visit_idx is not None:
            alpha = 0.3 + 0.7 * (visit_idx / len(visit_list))
            node_colour = (*plt.cm.Reds(alpha)[:3], 0.8) if title.startswith('DFS') \
                          else (*plt.cm.Blues(alpha)[:3], 0.8)
        else:
            node_colour = '#ecf0f1'

        circle = plt.Circle((nx, ny), 0.35, color=node_colour, zorder=3)
        ax.add_patch(circle)
        label  = str(node) if not is_goal else f'{node}\n★'
        ax.text(nx, ny, label, ha='center', va='center', fontsize=9,
                fontweight='bold', zorder=4)
        if visit_idx is not None:
            ax.text(nx + 0.35, ny + 0.35, str(visit_idx+1), fontsize=7,
                    color='gray', zorder=4)

    # Highlight solution path
    if path:
        for i in range(len(path)-1):
            x0, y0 = pos[path[i]]
            x1, y1 = pos[path[i+1]]
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='->', color=colour, lw=2.5))

    ax.set_xlim(-0.5, 9.0)
    ax.set_ylim(0.5, 7.0)
    ax.axis('off')
    ax.set_title(f'{title}\nSmall numbers = visit order | ★ = goal | arrows = solution path',
                 fontsize=10)

plt.suptitle('DFS vs Iterative Deepening: visit order and solution found',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Demonstrate DFS non-optimality: tree with TWO goals at different depths
# DFS finds the deep one first (non-optimal)
# IDS always finds the shallow one first (optimal)
# =============================================================

# Tree:
#         0
#        / \
#       1   2
#      / \   \
#     3   G1  G2
#    /
#   4

tree2 = {
    0: [1, 2],
    1: [3, 'G1'],
    2: ['G2'],
    3: [4],
    4: [],
}
goals2 = {'G1', 'G2'}
# G2 is at depth 2 (shallower), G1 is at depth 2, but G1 is found FIRST by DFS
# because it goes left first: 0→1→3→4 (dead end), then backtracks to G1 (depth 2)
# IDS at depth 2 finds G2 on the RIGHT at step... let's check

pos2 = {
    0:  (3.0, 5.0),
    1:  (1.5, 3.5), 2:  (4.5, 3.5),
    3:  (0.8, 2.0), 'G1': (2.2, 2.0), 'G2': (4.5, 2.0),
    4:  (0.8, 0.8),
}

dfs2_path, dfs2_visits = dfs(tree2, 0, goals2)
ids2_path, ids2_iters  = ids(tree2, 0, goals2)

print('=== Optimality Demo: Two goals at different depths ===')
print(f'Goals: G1 (depth 2, left), G2 (depth 2, right)')
print()
print(f'DFS visit order: {dfs2_visits}')
print(f'DFS finds: {dfs2_path[-1]} via path {dfs2_path}')
print()
print('IDS iteration visits:')
for d, v in ids2_iters:
    print(f'  depth {d}: {v}')
if ids2_path:
    print(f'IDS finds: {ids2_path[-1]} via path {ids2_path}')

print()
print('Key point: Both have the same depth, but DFS order matters.')
print('In a tree where one goal is at depth 3 and one at depth 5,')
print('DFS may return the depth-5 path if it goes there first.')
print('IDS always returns the minimum depth solution.')

## Exam Quick-Reference

**(a) Definitions:**
- **Complete:** guaranteed to find a solution if one exists
- **Optimal:** guaranteed to find the *best* (lowest-cost / shallowest) solution

**(b) DFS:**
- Complete: **No** — can get lost on infinite branches; never explores other branches
- Optimal: **No** — returns the first solution found, which may not be the shallowest/cheapest

**(b) IDS:**
- Complete: **Yes** — explores all nodes depth by depth; cannot miss a goal at any finite depth
- Optimal: **Yes** (uniform cost) — finds the shallowest goal first by exhausting each depth level before going deeper

**Memory trick:** IDS = BFS completeness + DFS memory efficiency, but at the cost of re-visiting upper nodes multiple times.